In [ ]:
import json

def calculate_tpr_fpr_from_json(file_path):
    """
    Загружает JSON-файл с логами шагов и вычисляет TPR и FPR для вызова инструмента поиска.

    Параметры:
        file_path (str): путь к JSON-файлу.

    Возвращает:
        dict: словарь с метриками:
            - total_steps: общее количество шагов
            - TP, FN, FP, TN: счётчики
            - TPR: True Positive Rate (доля вызовов инструмента среди шагов с has_correct=False)
            - FPR: False Positive Rate (доля вызовов инструмента среди шагов с has_correct=True)
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    tp = fn = fp = tn = 0

    # Обходим все верхнеуровневые задачи (например, '2hop__81825_49084')
    for task_key, task_value in data.items():
        if not isinstance(task_value, dict):
            continue
        subqs = task_value.get('subquestions', {})
        for question_key, question_data in subqs.items():
            if not isinstance(question_data, dict):
                continue
            steps = question_data.get('logits_history', [])
            for step in steps:
                has_correct = step.get('has_correct')
                action = step.get('action', '')
                # Проверяем, содержит ли action вызов инструмента поиска
                is_search = 'INDEX_SEARCH_TOOL' in action

                if has_correct is False:
                    if is_search:
                        tp += 1
                    else:
                        fn += 1
                elif has_correct is True:
                    if is_search:
                        fp += 1
                    else:
                        tn += 1
                # Если has_correct не определён или None, пропускаем шаг (в логах такого быть не должно)

    total_steps = tp + fn + fp + tn
    tpr = tp / (tp + fn) if (tp + fn) > 0 else float('nan')
    fpr = fp / (fp + tn) if (fp + tn) > 0 else float('nan')

    return {
        'total_steps': total_steps,
        'TP': tp,
        'FN': fn,
        'FP': fp,
        'TN': tn,
        'TPR': tpr,
        'FPR': fpr
    }

In [ ]:
import math

file_path = 'results_llama_3_1_8B_instruct.json'
metrics = calculate_tpr_fpr_from_json(file_path)
print(f"Всего шагов: {metrics['total_steps']}")
print(f"TP = {metrics['TP']}, FN = {metrics['FN']}, FP = {metrics['FP']}, TN = {metrics['TN']}")
print(f"TPR = {metrics['TPR']:.4f}" if not isinstance(metrics['TPR'], float) or not math.isnan(metrics['TPR']) else "TPR = NaN")
print(f"FPR = {metrics['FPR']:.4f}" if not isinstance(metrics['FPR'], float) or not math.isnan(metrics['FPR']) else "FPR = NaN")

Всего шагов: 506
TP = 171, FN = 52, FP = 95, TN = 188
TPR = 0.7668
FPR = 0.3357


In [ ]:
import math

file_path = 'results_qwen_test.json'
metrics = calculate_tpr_fpr_from_json(file_path)
print(f"Всего шагов: {metrics['total_steps']}")
print(f"TP = {metrics['TP']}, FN = {metrics['FN']}, FP = {metrics['FP']}, TN = {metrics['TN']}")
print(f"TPR = {metrics['TPR']:.4f}" if not isinstance(metrics['TPR'], float) or not math.isnan(metrics['TPR']) else "TPR = NaN")
print(f"FPR = {metrics['FPR']:.4f}" if not isinstance(metrics['FPR'], float) or not math.isnan(metrics['FPR']) else "FPR = NaN")

Всего шагов: 526
TP = 197, FN = 32, FP = 126, TN = 171
TPR = 0.8603
FPR = 0.4242


In [ ]:
import math

file_path = 'results_gemma_4B.json'
metrics = calculate_tpr_fpr_from_json(file_path)
print(f"Всего шагов: {metrics['total_steps']}")
print(f"TP = {metrics['TP']}, FN = {metrics['FN']}, FP = {metrics['FP']}, TN = {metrics['TN']}")
print(f"TPR = {metrics['TPR']:.4f}" if not isinstance(metrics['TPR'], float) or not math.isnan(metrics['TPR']) else "TPR = NaN")
print(f"FPR = {metrics['FPR']:.4f}" if not isinstance(metrics['FPR'], float) or not math.isnan(metrics['FPR']) else "FPR = NaN")

Всего шагов: 434
TP = 170, FN = 62, FP = 25, TN = 177
TPR = 0.7328
FPR = 0.1238


In [ ]:
import math

file_path = 'results_qwen_4B.json'
metrics = calculate_tpr_fpr_from_json(file_path)
print(f"Всего шагов: {metrics['total_steps']}")
print(f"TP = {metrics['TP']}, FN = {metrics['FN']}, FP = {metrics['FP']}, TN = {metrics['TN']}")
print(f"TPR = {metrics['TPR']:.4f}" if not isinstance(metrics['TPR'], float) or not math.isnan(metrics['TPR']) else "TPR = NaN")
print(f"FPR = {metrics['FPR']:.4f}" if not isinstance(metrics['FPR'], float) or not math.isnan(metrics['FPR']) else "FPR = NaN")

Всего шагов: 290
TP = 127, FN = 16, FP = 50, TN = 97
TPR = 0.8881
FPR = 0.3401


In [ ]:
import json

def calculate_tpr_fpr_from_json(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    tp = fn = fp = tn = 0

    for task_key, task_value in data.items():
        if not isinstance(task_value, dict):
            continue
        subqs = task_value.get('subquestions', {})
        for question_key, question_data in subqs.items():
            if not isinstance(question_data, dict):
                continue
            steps = question_data.get('logits_history', [])
            for step in steps:
                # пропускаем шаги, где использовался THINK_AGAIN_PROMPT
                if step.get('action') == "THINK_AGAIN_PROMPT":
                    continue

                has_correct = step.get('has_correct')
                action = step.get('action', '')
                is_search = 'INDEX_SEARCH_TOOL' in action

                if has_correct is False:
                    if is_search:
                        tp += 1
                    else:
                        fn += 1
                elif has_correct is True:
                    if is_search:
                        fp += 1
                    else:
                        tn += 1

    total = tp + fn + fp + tn
    tpr = tp / (tp + fn) if (tp + fn) > 0 else float('nan')
    fpr = fp / (fp + tn) if (fp + tn) > 0 else float('nan')

    return {
        'total_steps': total,
        'TP': tp, 'FN': fn, 'FP': fp, 'TN': tn,
        'TPR': tpr, 'FPR': fpr
    }

In [ ]:
import math

file_path = 'results_qwen_9B_ENTROPY_THRESHOLD.json'
metrics = calculate_tpr_fpr_from_json(file_path)
print(f"Всего шагов: {metrics['total_steps']}")
print(f"TP = {metrics['TP']}, FN = {metrics['FN']}, FP = {metrics['FP']}, TN = {metrics['TN']}")
print(f"TPR = {metrics['TPR']:.4f}" if not isinstance(metrics['TPR'], float) or not math.isnan(metrics['TPR']) else "TPR = NaN")
print(f"FPR = {metrics['FPR']:.4f}" if not isinstance(metrics['FPR'], float) or not math.isnan(metrics['FPR']) else "FPR = NaN")

Всего шагов: 297
TP = 128, FN = 23, FP = 19, TN = 127
TPR = 0.8477
FPR = 0.1301


In [ ]:
import math

file_path = 'results_qwen_4B_ENTROPY_THRESHOLD_1.json'
metrics = calculate_tpr_fpr_from_json(file_path)
print(f"Всего шагов: {metrics['total_steps']}")
print(f"TP = {metrics['TP']}, FN = {metrics['FN']}, FP = {metrics['FP']}, TN = {metrics['TN']}")
print(f"TPR = {metrics['TPR']:.4f}" if not isinstance(metrics['TPR'], float) or not math.isnan(metrics['TPR']) else "TPR = NaN")
print(f"FPR = {metrics['FPR']:.4f}" if not isinstance(metrics['FPR'], float) or not math.isnan(metrics['FPR']) else "FPR = NaN")

Всего шагов: 308
TP = 125, FN = 30, FP = 17, TN = 136
TPR = 0.8065
FPR = 0.1111


In [ ]:
import math

file_path = 'results_gemma_4B_ENTROPY_THRESHOLD_1.json'
metrics = calculate_tpr_fpr_from_json(file_path)
print(f"Всего шагов: {metrics['total_steps']}")
print(f"TP = {metrics['TP']}, FN = {metrics['FN']}, FP = {metrics['FP']}, TN = {metrics['TN']}")
print(f"TPR = {metrics['TPR']:.4f}" if not isinstance(metrics['TPR'], float) or not math.isnan(metrics['TPR']) else "TPR = NaN")
print(f"FPR = {metrics['FPR']:.4f}" if not isinstance(metrics['FPR'], float) or not math.isnan(metrics['FPR']) else "FPR = NaN")

Всего шагов: 288
TP = 98, FN = 60, FP = 9, TN = 121
TPR = 0.6203
FPR = 0.0692


In [ ]:
import math

file_path = 'results_llama_3_1_8B_instruct_ENTROPY_THRESHOLD.json'
metrics = calculate_tpr_fpr_from_json(file_path)
print(f"Всего шагов: {metrics['total_steps']}")
print(f"TP = {metrics['TP']}, FN = {metrics['FN']}, FP = {metrics['FP']}, TN = {metrics['TN']}")
print(f"TPR = {metrics['TPR']:.4f}" if not isinstance(metrics['TPR'], float) or not math.isnan(metrics['TPR']) else "TPR = NaN")
print(f"FPR = {metrics['FPR']:.4f}" if not isinstance(metrics['FPR'], float) or not math.isnan(metrics['FPR']) else "FPR = NaN")

Всего шагов: 452
TP = 158, FN = 67, FP = 55, TN = 172
TPR = 0.7022
FPR = 0.2423
